# Fraud Detection MLOps — narrativa técnica

Autor: Wanderson Ferreira. Síntese atualizada em 2026-10-09.

Laboratório reproduzível de MLOps para priorização de investigação de fraude,
com histórico temporal, feedback atrasado e evidências auditáveis. Este notebook
reúne decisões e marcos; as seções de preparação preservam seu estado histórico.
O estado atual está na síntese e no README, não nas pendências de marcos anteriores.
**[Mural de todas as metas e próximos passos](../docs/project/ROADMAP.md).**

**HGB = Histogram-based Gradient Boosting**: boosting de árvores baseado em
histogramas, implementado por `HistGradientBoostingClassifier`. É a referência atual.
Investigação detalhada fica nos notebooks por etapa e procedimentos em `docs/`,
conforme a [política](../docs/project/DOCUMENTATION_POLICY.md).


## Problema e direção

**Com capacidade limitada de investigação, quais clientes devemos priorizar usando
somente a informação disponível naquele momento, e como manter essa decisão
confiável quando dados e padrões mudam?**

Uma transação é um pagamento realizado por um cliente num terminal. Terminal
representa o ponto de aceitação do pagamento; uma maquininha é um exemplo intuitivo.
Cliente, terminal e transação têm identificadores distintos. O mesmo cliente pode
realizar vários pagamentos, e vários clientes podem usar o mesmo terminal.

O sistema antifraude combina controles de segurança, regras de bloqueio e
pontuação, modelo orientado por dados e investigadores. Nosso HGB ocupa o papel
do modelo que ordena risco para apoiar a triagem. Score, alerta e fraude confirmada
têm significados diferentes: investigar um alerta fornece feedback posterior.
[Definições](../docs/project/PROBLEM_CONTEXT.md#antes-dos-modelos-cliente-transação-e-terminal)
e [camadas do sistema](../docs/project/PROBLEM_CONTEXT.md#onde-nosso-modelo-se-encaixa).

O [contexto do problema](../docs/project/PROBLEM_CONTEXT.md) liga marcos históricos de fraude
e ML à investigação atual, com referências primárias. A proposta conecta qualidade
do ranking, política de triagem e confiabilidade operacional. Nosso ponto de
partida é cartões simulados pelo Handbook; o modelo não foi validado para dados
bancários reais ou golpes no Pix.

A referência foi congelada antes da avaliação final, que consumiu o teste de maio:
AP 0,640703 e precisão diária @100 de 55%. O estudo Optuna de 20 trials conservou
a referência. A API de serving tem checks sintéticos e a suíte do autor aprovou
302 testes; exportação/HTTP da referência real e Docker ainda precisam de execução.
O marco foi integrado pelo [PR #1](https://github.com/wanderson42/fraud-detection-mlops/pull/1).
O protocolo fixa 02–15/09 para avaliação e 16–30/09 para replay. A CLI de plano
lê somente JSONs; o executor auditável foi implementado e conferido com dados
sintéticos. A avaliação nativa de setembro ainda está pendente. Após revisar seus
resultados, retomaremos exportação real, Docker, Prefect e replay causal. Replay/streaming, observabilidade e promoção têm marcos próprios.
A [síntese para stakeholders](../docs/project/STAKEHOLDERS.md) explica quem pode usar o
laboratório e quais evidências sustentam seu estado atual.


## Marco 1 — Bronze reproduzível

Implementação: [`18242fc`](https://github.com/wanderson42/fraud-detection-mlops/commit/18242fcc8e340bad52a394c7c3624b78bd809b6c).

Decisão central: fixar uma revisão do Handbook e preservar seus bytes originais,
com inventário, checksums, manifesto e auditorias. Isso estabelece uma base
rastreável para a futura transformação ELT.

O autor informou a aquisição completa e a verificação de **183/183 arquivos**,
cobrindo 2018-04-01 a 2018-09-30. A execução completa reutilizou sete arquivos e
baixou os 176 restantes. Foram informados **39 testes locais aprovados**; a
[CI dessa revisão](https://github.com/wanderson42/fraud-detection-mlops/actions/runs/37514553686)
foi consultada e está aprovada.

Naquele marco, a validação cobria aquisição e integridade; o diagnóstico semântico
foi realizado na etapa seguinte.
Fontes: [recibo](../references/evidence/bronze_2026-10-06.json) e
[notebook da Bronze](stages/01_bronze_ingestion.ipynb).


## Marco 2 — diagnóstico completo e contrato da Silver

O autor informou o diagnóstico dos 183 arquivos: **1.754.155 transações**, com o
mesmo número de IDs distintos, 14.681 fraudes e 1.739.474 transações genuínas.
Os controles implementados não apontaram nulos, duplicatas ou inconsistências
temporais/de rótulos. Foram encontrados **42 valores monetários zero**.
O [recibo](../references/evidence/silver_profile_2026-10-06.json) transcreve a saída
do terminal; não é um output de execução deste notebook.

Decisão: preservar todos os registros, inclusive zeros e extremos, e converter
as nove colunas para tipos explícitos. A política se apoia no diagnóstico e no
arredondamento do gerador publicado. Consulte o [contrato](../docs/data/SILVER_CONTRACT.md)
e o [notebook da Silver](stages/02_silver_data_contract.ipynb).

O autor também concluiu a construção e a verificação da **Silver `silver_v1`**:
183 partições Parquet, com todas as contagens reconciliadas e os 42 zeros
preservados. Foram informados 70 testes locais aprovados em 0,91 s. A execução
responsável é `f9fc5a5083f14d9fadf6892a531e1488`, registrada no
[recibo da Silver](../references/evidence/silver_build_2026-10-06.json).

Os resultados são saídas de terminal fornecidas pelo autor; os arquivos nativos
permanecem locais. A implementação foi publicada em `acf1516`, com
[CI aprovada](https://github.com/wanderson42/fraud-detection-mlops/actions/runs/37563463479).
A próxima etapa usa a Silver para EDA e definição do protocolo temporal; a Gold
conterá as features, com testes contra vazamento.


## Validação de código — integração tox–Poetry

A configuração [tox.toml](../tox.toml) centraliza a sequência usada localmente e
na CI: verificar o lockfile, sincronizar dependências e executar Ruff/pytest em
`.tox/py314`. O Poetry instala as versões fixadas do lockfile nesse ambiente.
Uma checagem anterior à sincronização confirma o alvo e evita operar sobre a
`.venv` de desenvolvimento.

A integração foi validada no ambiente de preparação; a execução local do autor
e a CI desta alteração ainda aguardam aplicação/publicação. O escopo é o projeto
em modo editable a partir do checkout; um teste de wheel fora do checkout é um
contrato diferente. O processo completo está em [Testing](../docs/operations/TESTING.md).
Não há alteração dos dados ou resultados de modelo nessa melhoria.


## EDA da Silver e protocolo temporal — preparação de 2026-10-07

O autor informou validação local do tox: 70 testes em 1,57 s, com lint e formatação aprovados;
a execução completa levou 4,61 s. A nova CI dessa integração ainda não foi informada.

A [EDA](../docs/data/EDA.md) preparada usa somente o treino de 1 a 28 de abril.
O [protocolo temporal](../docs/modeling/EVALUATION_PROTOCOL.md) reserva validação e teste final,
com sete dias de feedback entre as janelas. O teste final permanece fora da seleção inicial;
a futura Gold implementará as features e os splits.

O [notebook da etapa](stages/03_silver_eda.ipynb) orienta a leitura dos relatórios locais.
A execução real e seus achados permanecem pendentes; o
[recibo](../references/evidence/eda_preparation_2026-10-07.json) registra apenas a preparação.
Não há modelo avaliado nem nova métrica de detecção.


## EDA de treino informada pelo autor — 2026-10-07

Execução `733686de23204cd6b9a5a1ec1cfbc2e7`: nove outputs verificados, 268.668 transações,
1.505 fraudes (0,5602%), quatro valores zero preservados e IDs distintos.
A evidência é [saída de terminal compartilhada](../references/evidence/eda_training_2026-10-07.json).
O autor compartilhou quatro tabelas agregadas; os arquivos nativos e o gráfico não foram inspecionados diretamente.

> O desbalanceamento já mostra por que **acurácia não será nossa métrica principal**: prever todas as transações como genuínas produziria aproximadamente **99,44% de acurácia**, com **recall de fraude igual a zero**. Isso sustenta a escolha de Average Precision para avaliar o ranking.

O cálculo é um contrafactual da classe majoritária no treino, sem modelo ajustado nem uso de holdouts.
A prevalência cresce entre os quatro blocos semanais; os valores de fraude apresentam cauda maior,
com ocorrências de valor baixo. A hipótese inicial para Gold é combinar valor atual e históricos
causais de cliente/terminal, tratando histórico insuficiente e atraso dos rótulos.

Os [achados e hipóteses](../docs/data/EDA.md#achados-informados-pelo-autor-em-2026-10-07)
estão ligados ao [notebook da etapa](stages/03_silver_eda.ipynb).
Commit do código avaliado, checks locais da EDA e CI correspondente ainda não foram informados.
Não há modelo avaliado; a Gold é a próxima implementação.


## Gold causal — preparação de 2026-10-07

A EDA e a integração tox foram publicadas em `fc22a48`, com
[CI aprovada](https://github.com/wanderson42/fraud-detection-mlops/actions/runs/37634000505).
A [Gold](../docs/data/GOLD_CONTRACT.md) preparada materializa 19 preditores nas três janelas
do protocolo, preservando linhas e separando alvo/IDs dos preditores.

Históricos de cliente/terminal usam passado estrito de 1 e 7 dias. Rótulos conhecidos
usam janelas deslocadas em sete dias; peers são excluídos e históricos vazios são tratados
com suporte explícito e flags de validade. Não há transformações aprendidas nesta etapa.

O [notebook da Gold](stages/04_gold_temporal_features.ipynb) e o
[recibo](../references/evidence/gold_preparation_2026-10-07.json) descrevem decisões e testes
controlados. Construção real e CI da Gold estão pendentes; ainda não há modelo avaliado.


## Gold real — execução informada em 2026-10-07

O autor forneceu build com status `success` e verify concordante: **42 partições,
19 preditores e 402.877 linhas**. Treino: 268.668; validação: 67.255; teste: 66.954.
O treino reconcilia com a EDA anterior. Run de auditoria: `8b1564df49fc497080963986b05aa007`.

O [recibo da execução](../references/evidence/gold_build_2026-10-07.json)
transcreve o terminal e distingue essa evidência dos testes controlados. Os Parquets,
o manifesto e a auditoria nativos não foram inspecionados pelo assistente. Revisão `14ab57e` e CI aprovada foram conferidas.
A saída dos checks locais do autor ainda não foi informada. Os outputs salvos das células de verificação e leitura do treino
foram publicados pelo autor e preservados nesta atualização; o assistente não reexecutou o notebook.

Os 19 preditores são candidatos escolhidos no desenho do contrato. A seleção por
desempenho ficará no pipeline: ajuste no treino, comparação na validação e teste
final após congelar as escolhas. Não há modelo avaliado neste marco.


## Publicação e CI conferidas em 2026-10-07

Revisão [`14ab57e`](https://github.com/wanderson42/fraud-detection-mlops/commit/14ab57e15f0931ffb6966b26d390a42b514762b7),
com [CI aprovada](https://github.com/wanderson42/fraud-detection-mlops/actions/runs/37652509812):
104 testes em 11,37 s; lint, formatação e lockfile aprovados.
O [recibo](../references/evidence/gold_build_2026-10-07.json) associa
a construção real informada pelo autor à revisão publicada e à evidência da CI.

O notebook da Gold publicado contém verificação concordante e leitura do treino:
`X` (268.668, 19), `y` (268.668,) e metadados (268.668, 6). Seus outputs foram
conferidos e preservados; não foram gerados pela preparação desta documentação.
A CI usa dados controlados, e os artefatos nativos da Gold continuam locais.
Naquele marco nenhum modelo havia sido avaliado. O baseline posterior está abaixo.


## Baseline temporal — preparação de 2026-10-07

O [baseline](../docs/modeling/BASELINE.md) preparado compara um controle constante, regressão
logística com escala ajustada no treino e gradient boosting com configurações fixas.
AP da validação seleciona o candidato; ROC AUC e precisão diária por cliente são
secundárias. Nenhum score ou métrica de teste é produzido. A Gold permanece imutável.

O [notebook da etapa](stages/05_temporal_baseline.ipynb) lê uma execução explícita;
o [recibo](../references/evidence/baseline_preparation_2026-10-07.json) registra
somente os testes controlados da preparação. A execução real posterior está abaixo;
baseline/tracking publicados em `6ba10b5`, com CI aprovada. Os outputs anteriores são preservados.


## Baseline real e tracking — 2026-10-07

Execução do autor `f6d7aca720f74316b4183f97b6d866ab`: AP 0,623949 do gradient
boosting, contra 0,435001 da regressão. Precisão diária nos cem clientes: 54% e
48%, respectivamente. Validação: 67.255 transações/580 fraudes em sete dias;
teste final reservado. [Evidência](../references/evidence/baseline_validation_2026-10-07.json).

A [etapa MLflow/skops](../docs/operations/MLFLOW.md) preserva esses modelos em três runs
independentes, com pipelines completos e scores verificados após recarga. A integração
foi publicada e verificada pelo autor, sem refit ou teste.
[Evidência](../references/evidence/mlflow_execution_2026-10-07.json). Backend SQLite
e artefatos locais; Prefect, análise estatística e quality gate terão entregas próprias.
[Notebook da etapa](stages/06_mlflow_tracking.ipynb).


## Simplificação: tracking nativo e organização

A migração histórica foi concluída e retirada do código ativo. Novos treinamentos
usam `baseline_v2`, pipelines skops e logging nativo MLflow, com a mesma política
científica. Os testes de modelagem espelham o pacote e usam fixtures compartilhadas.
Os scaffolds vazios saíram; SHA256/JSON têm uma interface comum. Os modelos e os
outputs reais anteriores permanecem preservados. [Arquitetura](../docs/ARCHITECTURE.md)
e [fluxo atual](stages/06_mlflow_tracking.ipynb).
A preparação usa fixtures sintéticas; não representa novo resultado real do modelo.


## Diagnóstico da baseline — execução informada em 2026-10-07

O autor executou e verificou `961bb0c32fb84433af922908ea58e76b`: sete outputs,
reconciliação de scores e notebook da [etapa 07](stages/07_baseline_diagnostics.ipynb)
atualizado. O [recibo](../references/evidence/diagnostics_execution_2026-10-07.json)
transcreve terminal e tabelas; os arquivos nativos e esse notebook atualizado
não foram enviados nesta etapa. Últimos checks locais: 142 testes aprovados,
29 avisos externos, lint, formatação e lockfile aprovados.

Em sete dias e 700 posições de alerta, HGB priorizou 378 ocorrências fraudulentas
de cliente/dia, contra 336 da regressão: **42 ocorrências adicionais ao mesmo
orçamento**. Não são pessoas únicas, perdas evitadas ou decisões de uma fila online.
A política atual ordena clientes retrospectivamente usando o dia completo.

Permutação indica forte dependência de valor, risco conhecido do terminal e desvio
de gastos. São sinais compatíveis com o gerador, inclusive sua regra artificial
de valor acima de 220. SHAP descreve contribuições ao score em log-odds e pode
ordenar features de modo diferente da dependência da AP. Importância negativa por
permutação motiva hipóteses, não remoção automática de features.

[Interpretação completa](../docs/modeling/DIAGNOSTICS.md#resultados).
O próximo protocolo fixará hipóteses, ganho relevante e orçamento antes de ablações
ou tuning. Não há teste de hipótese executado nem prova estatística de superioridade.
Os modelos continuam congelados; o teste final permanece reservado.


## Experimentos controlados — protocolo preparado em 2026-10-08

O [protocolo `experiment_v1`](../docs/modeling/EXPERIMENT_PROTOCOL.md) define três ablações
do HGB: remover volume do terminal, contagens de fraudes conhecidas e ambos.
São 17, 17 e 15 preditores, com os mesmos dados e hiperparâmetros. A referência
histórica será reutilizada; Gold e teste reservado permanecem com seus contratos.

O gate prático exige ganho absoluto de AP ≥ 0,01 e precisão diária nos 100
clientes ≥ 0,02 frente à referência. São escolhas deste projeto para revisão de
congelamento, sem promoção automática. Comparações diárias pareadas e exclusão
de um dia por vez mostrarão influência temporal; sete dias não sustentam aqui
uma declaração formal de superioridade estatística.

Estado: **política definida; runner e resultados pendentes**. Congelar o protocolo
no Git antes dos fits. O [notebook da etapa](stages/08_controlled_experiments.ipynb)
permite ler o catálogo; receberá resultados reais na próxima implementação.


## Ablações controladas — execução e decisão de 2026-10-08

O autor executou e verificou `21ccedf10d944092ba874153c1d21257` sobre `de41ee0`,
com 18 artefatos e três candidatos. [Recibo](../references/evidence/controlled_ablation_execution_2026-10-08.json)
e [notebook da etapa](stages/08_controlled_experiments.ipynb).

O gate preservou o HGB de 19 features. A referência priorizou 378 ocorrências
fraudulentas de cliente/dia em 700 vagas; as ablações priorizaram 374, 373 e 372.
O candidato de 15 features aumentou AP para 0,628595 e reduziu precisão diária
para 53,14%. Seu ganho de AP fica abaixo de +0,01 nas sete exclusões de dia;
a precisão operacional piora em todas. A remoção de contagens muda o sinal do
pequeno efeito de AP em duas exclusões.

A evidência de retreinamento orienta conservar as features sob este gate. O
catálogo foi encerrado sem alterar os critérios após observar resultados.
Não há hipótese confirmatória, refit com validação, promoção em produção ou
avaliação do teste final. As contagens são cliente/dia, não pessoas únicas nem
perdas financeiras evitadas. [Decisão completa](../docs/modeling/EXPERIMENT_PROTOCOL.md#resultados-e-decisão--2026-10-08).


## Congelamento da referência — preparação de 2026-10-08

O primeiro catálogo de ablações conserva o HistGradientBoostingClassifier (HGB)
com 19 features. A etapa `modeling.experiments.candidate_freeze` registra a referência existente,
seus parâmetros, identidade MLflow/skops, ambiente e política de 100 clientes
por dia. Confere os scores de validação sem refit ou leitura do teste.

A política final propõe AP ≥ 0,50 e precisão diária @100 ≥ 0,45 para seguir ao
serving de laboratório. São metas práticas, definidas após a validação e antes
do teste, sem estimativa de retorno financeiro ou autorização para produção.
O ranking diário usa o dia completo retrospectivo; não equivale a alertas online.

[Contrato e procedimento](../docs/modeling/EVALUATION_PROTOCOL.md#congelamento-antes-da-avaliação-final).
O autor deve versionar a implementação, executar o congelamento local e
versionar `references/frozen_candidate_v1.json` antes da avaliação final.
**Congelamento real e teste permanecem pendentes.** Não é necessário repetir a baseline.


## Checkpoint posterior — congelamento concluído e avaliação final preparada

O checkpoint de preparação anterior foi concluído pelo autor: implementação
`93e6958`, recibo versionado em `623dc86` e verificação com `--require-committed`.
HGB com 19 features; SHA256 do recibo
`6281337ac9ad872947470d94de0307d7c1934365538877376559d3dee2e57120`.
[Execução do congelamento](../references/evidence/freeze_execution_2026-10-08.json).

A próxima etapa é [09 — avaliação final](stages/09_final_evaluation.ipynb): mesmo
modelo, apenas pontuação do teste, controle constante, métricas diárias e Model
Card. AP ≥ 0,50 e precisão diária @100 ≥ 0,45 permanecem os critérios definidos.
A implementação deve ser validada e commitada antes da execução local.
**Execução do teste e resultado do gate ainda pendentes.**

[Procedimento](../docs/modeling/EVALUATION_PROTOCOL.md#executar-a-avaliação-final).


## Checkpoint — avaliação final concluída pelo autor

Na revisão `e0fddc0`, o candidato congelado foi pontuado sem refit e os cinco
outputs foram verificados localmente pelo autor. Teste de 20–26/05/2018:
66.954 transações e 597 fraudes (0,892%).

| Métrica | HGB congelado | Controle constante |
| --- | ---: | ---: |
| AP | **0,640703** | 0,008917 |
| ROC AUC | **0,890356** | 0,500000 |
| Precisão diária por cliente @100 | **55%** | 1,857% |

Os critérios AP ≥ 0,50 e precisão diária @100 ≥ 0,45 foram atingidos.
Decisão: `eligible_for_laboratory_serving_review`; sem promoção em produção.
[Execução informada](../references/evidence/final_evaluation_execution_2026-10-08.json)
e [Model Card curada](../docs/modeling/MODEL_CARD.md). Esta célula sintetiza saídas de
terminal; não representa execução de células do notebook pelo assistente.

**O teste foi consumido.** Melhorias futuras exigem outra janela e protocolo.
O CSV diário fornecido foi revisado: recall médio de **73,02%**, precisão entre
**47% e 61%**, 385 ocorrências de cliente com fraude na fila e 142 fora, entre
527 ocorrências de cliente por dia. Esses totais não são pessoas únicas.
A conferência direta dos bytes da Model Card gerada permanece pendente.
O [primeiro contrato de inferência](../docs/operations/SERVING_CONTRACT.md)
recebe 19 features calculadas e devolve score identificado. Serviço, Docker e
paridade serão demonstrados na próxima implementação, sem repetir a baseline.


## Contrato de integração de modelos

`model_interface_v1` formaliza como contribuir algoritmos que usam o mesmo treinamento, avaliação, persistência e MLflow. A implementação atual usa Pipelines scikit-learn, valida entradas e probabilidades e conserva a política fixa da baseline.

O [contrato e o exemplo](../docs/modeling/CONTRIBUTING_MODELS.md) explicam as responsabilidades. O exemplo Gaussian Naive Bayes roda somente sobre dados sintéticos; não representa outra baseline nem uma otimização concluída. [Testing](../docs/operations/TESTING.md#contrato-de-integração-de-modelos) registra o escopo dos checks. Optuna usa essa mesma interface no ciclo temporal; o relatório do autor concluiu 20 trials e conservou a referência. Veja a etapa 10 e o [mural](../docs/project/ROADMAP.md).

## 10 — Otimização temporal do HGB com Optuna

O relatório fornecido pelo autor concluiu 20 trials e 63 fits, sem falhas, nas
três janelas de julho–agosto. Nenhum trial passou os gates: `retain_reference`.
O melhor ganho médio de AP foi +0,005807; confirmação não avaliada e sem promoção.

[Protocolo e evidência](../docs/modeling/EVALUATION_PROTOCOL.md#resultado-da-busca-informado-pelo-autor)
e [leitura dos artefatos](stages/10_hgb_optuna.ipynb). O teste de maio está consumido;
setembro continua reservado. Uma nova hipótese exige protocolo próprio.


## 11 — Organização para evoluir a infraestrutura

Contratos, algoritmos e experimentos têm subpacotes próprios. Métricas/diagnósticos
ficam em `evaluation/`; MLflow/skops em `integrations/`. Os executores compartilham
`candidate_training`, sem depender do runner da baseline. Comandos antigos têm
compatibilidade de transição; evidências históricas conservam suas identidades.

[Etapa 11](stages/11_modeling_organization.ipynb), [arquitetura](../docs/ARCHITECTURE.md)
e [índice da documentação](../docs/README.md). Dados/features terão outra etapa;
serving e empacotamento preservam os contratos e exigem verificação própria.


## Organização de dados e features

A segunda etapa separa contratos, ingestão, datasets e qualidade em `data/`;
`features/` contém schema, cálculo causal e validação. Imports e comandos antigos
continuam disponíveis. A conferência usa dados sintéticos antes/depois, preservando
schemas, valores e causalidade. Os dados reais e a reserva confirmatória não são
abertos nesta preparação.

[Etapa 12](stages/12_data_features_organization.ipynb),
[arquitetura](../docs/ARCHITECTURE.md#dados-e-features) e
[migração](../docs/operations/OPERATIONS.md#migração-de-dados-e-features).


## 13 — Serving de laboratório: preparação de 2026-10-09

A API recebe 19 features e devolve score com identidade do modelo/release. A
exportação preserva os bytes congelados e confere a validação após JSON sem fit
ou nova pontuação do teste. Carga e HTTP foram conferidos com dados sintéticos;
o wheel respondeu fora do checkout sem a stack de experimentação.

O autor informou 302 testes aprovados, 89 avisos e tox `py314` aprovado
([relato](../references/evidence/laboratory_serving_author_validation_2026-10-09.json)).
Falta a exportação local da referência e a entrega Docker com medições. O protocolo
estatístico é a próxima etapa após integrar este marco. A reserva de setembro
permanece fechada. [Etapa](stages/13_laboratory_serving.ipynb),
[contrato e comandos](../docs/operations/SERVING_CONTRACT.md) e
[evidência](../references/evidence/laboratory_serving_preparation_2026-10-09.json).


## 14 — Protocolo estatístico da referência: preparação de 2026-10-09

Nenhum trial Optuna foi elegível. Fixamos a pergunta sobre o modelo já congelado
de abril em 02–15/09, sem refit ou calibração. A nova CLI prepara o plano lendo
somente quatro JSONs; não pontua dados nem abre tracking. 16–30/09 continua
reservado para replay histórico com feedback atrasado.

O cálculo da fila aleatória condiciona nas populações e rótulos de cada dia.
Seu intervalo de capturas e sua probabilidade de cauda descrevem essa política
hipotética; não são incerteza da qualidade futura do modelo. Inferência temporal
confirmatória ainda exige candidato, hipótese e condições justificadas.

[Etapa 14](stages/14_reference_assessment_protocol.ipynb),
[protocolo e comandos](../docs/modeling/EVALUATION_PROTOCOL.md#protocolo-estatístico-da-referência--v1)
e [evidência da preparação](../references/evidence/reference_assessment_preparation_2026-10-09.json).


## 15 — Executor da referência: preparação de 2026-10-09

Uma autorização de execução separada preserva o protocolo estatístico v1 e a CLI
de plano. O executor verifica o modelo congelado antes das partições, registra
acesso/tentativas e usa somente 19/08–15/09 para produzir features causais e avaliar
02–15/09. Os 14 dias de resultados são publicados com hashes e podem ser
recalculados sem modelo, tracking ou Silver. Retomadas completas reutilizam scores.

Os checks são sintéticos, incluindo carga real de MLflow/skops num store temporário,
falhas, cobertura e dias de uma classe. Nenhum artefato nativo do autor foi
carregado nesta preparação. Teste confirmatório, replay e promoção continuam fora
desta execução. Próximo passo: validar/commitar o incremento, executar no Alienware
e revisar a evidência antes de retomar a infraestrutura.

[Etapa 15](stages/15_reference_assessment_execution.ipynb),
[execução e recuperação](../docs/modeling/EVALUATION_PROTOCOL.md#executor-auditável-da-janela-fixada)
e [recibo](../references/evidence/reference_assessment_execution_preparation_2026-10-09.json).
